# Module 3: Physics-Informed Digital Twin Evaluation

Compare the physics-informed surrogate model against a data-only baseline.
Key question: do physics constraints (point kinetics, energy conservation) improve predictions?

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from pathlib import Path

from src.data.dataset_twin import DigitalTwinDataset
from src.models.digital_twin.surrogate import ReactorSurrogate
from src.models.digital_twin.loss import PhysicsInformedLoss
from src.models.digital_twin.physics import (
    FEATURE_MAP, DEFAULT_PARAMS, point_kinetics_rhs, reactivity
)
from src.utils.config import Config
from data.scripts.preprocess import ACCIDENT_TYPES

sns.set_theme(style='whitegrid')
CHECKPOINT_DIR = Path('../checkpoints')

inv_map = {v: k for k, v in ACCIDENT_TYPES.items()}

PHYSICS_FEATURES = {
    'P': FEATURE_MAP.P, 'TAVG': FEATURE_MAP.TAVG,
    'THA': FEATURE_MAP.THA, 'THB': FEATURE_MAP.THB,
    'TCA': FEATURE_MAP.TCA, 'TCB': FEATURE_MAP.TCB,
    'WRCA': FEATURE_MAP.WRCA, 'WRCB': FEATURE_MAP.WRCB,
    'QMWT': FEATURE_MAP.QMWT, 'PWR': FEATURE_MAP.PWR,
    'PPM': FEATURE_MAP.PPM,
}

## 1. Load Models and Test Data

In [ ]:
# Load test dataset
test_ds = DigitalTwinDataset('../data/processed/test.pt', prediction_horizon=10)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

# Load scaler for inverse transform
with open('../data/processed/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
scaler_mean = torch.tensor(scaler.mean_, dtype=torch.float32)
scaler_std = torch.tensor(scaler.scale_, dtype=torch.float32)

def inverse_transform(x):
    return x * scaler_std + scaler_mean

# Load both models (use from_config to include loss_fn buffers in state_dict)
def load_model(ckpt_name, config_path):
    config = Config.from_yaml(config_path)
    model = ReactorSurrogate.from_config(config)
    ckpt = torch.load(CHECKPOINT_DIR / ckpt_name, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

model_physics = load_model('digital_twin_best.pt', '../configs/digital_twin.yaml')
model_baseline = load_model('digital_twin_no_physics_best.pt', '../configs/digital_twin_no_physics.yaml')

print(f'Test windows: {len(test_ds)}')
print(f'Context: {test_ds.context_size} steps, Horizon: 10 steps')
print(f'Features: {test_ds.n_features}')

## 2. Prediction Accuracy: MSE Comparison

In [ ]:
def compute_predictions(model, loader):
    all_preds, all_targets, all_contexts = [], [], []
    with torch.no_grad():
        for context, target in loader:
            pred = model(context)
            all_preds.append(pred)
            all_targets.append(target)
            all_contexts.append(context)
    return (
        torch.cat(all_preds),
        torch.cat(all_targets),
        torch.cat(all_contexts),
    )

pred_physics, targets, contexts = compute_predictions(model_physics, test_loader)
pred_baseline, _, _ = compute_predictions(model_baseline, test_loader)

# Overall MSE
mse_physics = F.mse_loss(pred_physics, targets).item()
mse_baseline = F.mse_loss(pred_baseline, targets).item()

print(f'Physics-informed MSE: {mse_physics:.6f}')
print(f'Data-only baseline MSE: {mse_baseline:.6f}')
print(f'Improvement: {(mse_baseline - mse_physics) / mse_baseline * 100:.1f}%')

In [ ]:
# MSE per prediction horizon step
mse_per_step_physics = ((pred_physics - targets) ** 2).mean(dim=(0, 2)).numpy()
mse_per_step_baseline = ((pred_baseline - targets) ** 2).mean(dim=(0, 2)).numpy()

fig, ax = plt.subplots(figsize=(10, 5))
steps = np.arange(1, len(mse_per_step_physics) + 1)
ax.plot(steps, mse_per_step_physics, 'o-', label='Physics-informed', color='#2196F3')
ax.plot(steps, mse_per_step_baseline, 's--', label='Data-only baseline', color='#FF5722')
ax.set_xlabel('Prediction Horizon (steps ahead)')
ax.set_ylabel('MSE (normalized space)')
ax.set_title('Prediction Error vs Horizon')
ax.legend()
ax.set_xticks(steps)
plt.tight_layout()
plt.show()

## 3. Physics Residual Comparison

Key result: does the physics-informed model produce predictions more consistent with point kinetics?

In [ ]:
def compute_physics_residual(predictions):
    """Compute point kinetics residual in physical space."""
    pred_phys = inverse_transform(predictions)
    fm = FEATURE_MAP
    
    n = pred_phys[:, :, fm.PWR]
    T_avg = pred_phys[:, :, fm.TAVG]
    ppm = pred_phys[:, :, fm.PPM]
    
    # Numerical dn/dt
    dn_dt_numerical = n[:, 1:] - n[:, :-1]  # dt=1
    # Analytical dn/dt
    dn_dt_physics = point_kinetics_rhs(n[:, :-1], T_avg[:, :-1], ppm[:, :-1])
    
    residual = (dn_dt_numerical - dn_dt_physics).abs()
    return residual

resid_physics = compute_physics_residual(pred_physics)
resid_baseline = compute_physics_residual(pred_baseline)

print(f'Mean physics residual (physics-informed): {resid_physics.mean():.4f}')
print(f'Mean physics residual (data-only): {resid_baseline.mean():.4f}')
print(f'Reduction: {(1 - resid_physics.mean() / resid_baseline.mean()) * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of residuals
axes[0].hist(resid_physics.flatten().numpy(), bins=50, alpha=0.7, label='Physics-informed', color='#2196F3')
axes[0].hist(resid_baseline.flatten().numpy(), bins=50, alpha=0.7, label='Data-only', color='#FF5722')
axes[0].set_xlabel('|Physics Residual|')
axes[0].set_ylabel('Count')
axes[0].set_title('Point Kinetics Residual Distribution')
axes[0].legend()

# Residual per horizon step
resid_per_step_p = resid_physics.mean(dim=0).numpy()
resid_per_step_b = resid_baseline.mean(dim=0).numpy()
steps = np.arange(1, len(resid_per_step_p) + 1)
axes[1].plot(steps, resid_per_step_p, 'o-', label='Physics-informed', color='#2196F3')
axes[1].plot(steps, resid_per_step_b, 's--', label='Data-only', color='#FF5722')
axes[1].set_xlabel('Prediction Step')
axes[1].set_ylabel('Mean |Physics Residual|')
axes[1].set_title('Physics Consistency vs Horizon')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Energy Conservation Violation Comparison

In [ ]:
def compute_conservation_violation(predictions):
    """Energy balance: Q vs W * Cp * (T_hot - T_cold)."""
    pred_phys = inverse_transform(predictions)
    fm = FEATURE_MAP
    
    Q = pred_phys[:, :, fm.QMWT]
    W = pred_phys[:, :, fm.WRCA] + pred_phys[:, :, fm.WRCB]
    T_hot = 0.5 * (pred_phys[:, :, fm.THA] + pred_phys[:, :, fm.THB])
    T_cold = 0.5 * (pred_phys[:, :, fm.TCA] + pred_phys[:, :, fm.TCB])
    
    Q_est = W * DEFAULT_PARAMS.Cp * (T_hot - T_cold) / 1000.0
    violation = (Q - Q_est).abs()
    return violation

cons_physics = compute_conservation_violation(pred_physics)
cons_baseline = compute_conservation_violation(pred_baseline)

print(f'Mean conservation violation (physics-informed): {cons_physics.mean():.4f} MW')
print(f'Mean conservation violation (data-only): {cons_baseline.mean():.4f} MW')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(cons_physics.flatten().numpy(), bins=50, alpha=0.7, label='Physics-informed', color='#2196F3')
ax.hist(cons_baseline.flatten().numpy(), bins=50, alpha=0.7, label='Data-only', color='#FF5722')
ax.set_xlabel('|Energy Conservation Violation| (MW)')
ax.set_ylabel('Count')
ax.set_title('Energy Balance Violation Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Per-Accident-Type Prediction Accuracy

In [ ]:
import pandas as pd

accident_types = test_ds.accident_types

mse_by_type = {}
for cls_idx in sorted(accident_types.unique().tolist()):
    mask = accident_types == cls_idx
    name = inv_map.get(cls_idx, str(cls_idx))
    mse_p = ((pred_physics[mask] - targets[mask]) ** 2).mean().item()
    mse_b = ((pred_baseline[mask] - targets[mask]) ** 2).mean().item()
    mse_by_type[name] = {'Physics': mse_p, 'Data-only': mse_b}

df_mse = pd.DataFrame(mse_by_type).T
df_mse['Improvement (%)'] = ((df_mse['Data-only'] - df_mse['Physics']) / df_mse['Data-only'] * 100)

fig, ax = plt.subplots(figsize=(12, 8))
df_mse[['Physics', 'Data-only']].plot(kind='barh', ax=ax)
ax.set_xlabel('MSE')
ax.set_title('Prediction MSE by Accident Type')
plt.tight_layout()
plt.show()

print(df_mse.to_string(float_format='%.4f'))

## 6. Feature Trajectory Plots

Predicted vs actual trajectories for key physics variables.

In [ ]:
# Select sample scenarios
sample_indices = [0, 100, 500, 1000]
plot_features = ['PWR', 'TAVG', 'P', 'WRCA']
plot_feature_idx = [PHYSICS_FEATURES[f] for f in plot_features]

fig, axes = plt.subplots(len(sample_indices), len(plot_features), figsize=(20, 4 * len(sample_indices)))

for row, sample_idx in enumerate(sample_indices):
    ctx = inverse_transform(contexts[sample_idx])
    tgt = inverse_transform(targets[sample_idx])
    pred_p = inverse_transform(pred_physics[sample_idx])
    pred_b = inverse_transform(pred_baseline[sample_idx])
    
    acc_type = inv_map.get(test_ds.accident_types[sample_idx].item(), '?')
    
    for col, (feat_name, feat_idx) in enumerate(zip(plot_features, plot_feature_idx)):
        ax = axes[row, col]
        ctx_len = ctx.shape[0]
        horizon = tgt.shape[0]
        
        # Context (observed)
        t_ctx = np.arange(ctx_len)
        ax.plot(t_ctx, ctx[:, feat_idx].numpy(), 'k-', alpha=0.5, label='Context')
        
        # Target and predictions
        t_pred = np.arange(ctx_len, ctx_len + horizon)
        ax.plot(t_pred, tgt[:, feat_idx].numpy(), 'k-', linewidth=2, label='Actual')
        ax.plot(t_pred, pred_p[:, feat_idx].numpy(), '--', color='#2196F3', label='Physics')
        ax.plot(t_pred, pred_b[:, feat_idx].numpy(), '--', color='#FF5722', label='Data-only')
        
        ax.axvline(x=ctx_len - 0.5, color='gray', linestyle=':', alpha=0.5)
        
        if row == 0:
            ax.set_title(feat_name)
        if col == 0:
            ax.set_ylabel(acc_type)
        if row == 0 and col == len(plot_features) - 1:
            ax.legend(fontsize=8)

plt.suptitle('Feature Trajectories: Predicted vs Actual', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
summary = pd.DataFrame({
    'Physics-Informed': {
        'MSE (normalized)': mse_physics,
        'Mean Physics Residual': resid_physics.mean().item(),
        'Mean Conservation Violation (MW)': cons_physics.mean().item(),
    },
    'Data-Only Baseline': {
        'MSE (normalized)': mse_baseline,
        'Mean Physics Residual': resid_baseline.mean().item(),
        'Mean Conservation Violation (MW)': cons_baseline.mean().item(),
    },
}).T

summary.index.name = 'Model'
print(summary.to_string(float_format='%.6f'))